In [ ]:
import json
import os

import earthaccess
import pandas as pd
import shutil
import xarray as xr

from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from tqdm import tqdm

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
                files.append(os.path.join(filepath, file))
    return files

### Find files to filter

In [ ]:
old_dir = Path('/nas/cee-ice/data/SWOT/PIXC_vD/')
new_dir = '/nas/cee-ice/data/SWOT/PIXC_filtered_vD/'

In [ ]:
# Get all files
pixc_paths  = [str(p) for p in old_dir.rglob('*') if p.is_file()]
pixc_files = {path[-71:-3] for path in pixc_paths}
print(len(pixc_files))

# Get already filtered files
filter_paths = getFiles(new_dir)
filter_files = {path[-76:-8] for path in filter_paths}
print(len(filter_files))

In [ ]:
# Filter the first list by checking if its suffix is in that set

missing_from_filter = [path for path in pixc_files if path not in filter_files]

# print(f"Found {len(already_present)} matches.")
print(f"Still need to handle {len(missing_from_filter)} unique entries.")

In [ ]:
missing_set = set(missing_from_filter)

# Extract basename, strip extension, and check against the set
missing_paths = [
    path for path in pixc_paths 
    if os.path.splitext(os.path.basename(path))[0] in missing_set
]

In [ ]:
output_file = 'files_to_filter_vD_2026_03_28.json'

with open(output_file, 'w') as f:
    json.dump(missing_paths, f, indent=4)

print(f"Successfully saved {len(missing_from_filter)} entries to {output_file}")

In [ ]:
# Max job index
len(missing_from_filter) // 1000

### Find existing files

In [ ]:
swot_dir = '/nas/cee-ice/data/SWOT/PIXC_vD/'

pixc_paths = getFiles(swot_dir)
pixc_files = [path[-76:] for path in pixc_paths]
len(pixc_files)

### Check file health

In [ ]:
log_file_path = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC/logs/corrupt.txt'
bad_files = []

with open(log_file_path, 'r') as f:
    for line in f:
        if "Skipping physically corrupted file:" in line:
            # Split by the specific prefix and then by the trailing " |"
            path = line.split("Skipping physically corrupted file: ")[1].split(" |")[0]
            # Get just the .nc filename
            bad_files.append(os.path.basename(path))

bad_files = list(set(bad_files))
bad_paths = [swot_dir + file for file in bad_files]

In [ ]:
# Set to your target directory
target_dir = "." 

for filename in os.listdir(target_dir):
    if filename.startswith("SWOT_") and filename.endswith(".nc"):
        parts = filename.split('_')
        if len(parts) >= 6:
            pass_num = parts[5] # The 6th segment
            pass_dir = os.path.join(target_dir, pass_num)
            
            os.makedirs(pass_dir, exist_ok=True)
            shutil.move(os.path.join(target_dir, filename), os.path.join(pass_dir, filename))


In [ ]:
quarantine_dir = Path('/nas/cee-ice/data/SWOT/PIXC_vD_corrupt')
quarantine_dir.mkdir(exist_ok=True)

In [ ]:
for file_path in bad_paths:
    # Ensure file_path is stripped of any extra spaces or log text
    p = Path(file_path.strip()) 
    
    if p.exists():
        try:
            # shutil.move handles cross-filesystem moves and permissions better
            shutil.move(str(p), quarantine_dir / p.name)
            print(f"Moved: {p.name}")
        except Exception as e:
            print(f"Failed to move {p.name}: {e}")
    else:
        print(f"File not found: {p}")

In [ ]:
# # Redownload corrupted files
# earthaccess.login()

# def redownload_corrupted(bad_files, dest_root):
#     # Removed tqdm wrapper to avoid double progress bars
#     for full_path in bad_files:
#         # 1. Get the filename from the path
#         filename = os.path.basename(full_path)
        
#         # 2. Extract pass number for organized storage (6th field)
#         # SWOT_L2_HR_PIXC_039_315_... -> 315
#         pass_num = filename.split('_')[5]
#         target_dir = Path(dest_root) / pass_num
#         target_dir.mkdir(parents=True, exist_ok=True)
        
#         try:
#             # 3. Search for the specific granule by name
#             results = earthaccess.search_data(
#                 short_name='SWOT_L2_HR_PIXC_D', 
#                 granule_name=filename
#             )
            
#             if results:
#                 # 4. Download with threads=8 to the correct pass directory
#                 earthaccess.download(results, str(target_dir), threads=8)
#             else:
#                 print(f"Could not find {filename} in Earthdata archive.")
                
#         except Exception as e:
#             print(f"Failed to download {filename}: {e}")

In [ ]:
bad_files = getFiles(quarantine_dir)

In [ ]:
def get_granule_links(bad_files):
    all_links = []
    
    # 2. Wrap the list in tqdm() for a progress bar
    # 'desc' adds a label to the bar so you know what's happening
    for full_path in tqdm(bad_files, desc="Searching EarthData"):
        filename = os.path.basename(full_path)
        
        try:
            results = earthaccess.search_data(
                short_name='SWOT_L2_HR_PIXC_D', 
                granule_name=filename
            )
            
            if results:
                links = results[0].data_links()
                all_links.extend(links)
            else:
                # Use tqdm.write() so the print doesn't break the progress bar
                tqdm.write(f"Could not find {filename}")
                
        except Exception as e:
            tqdm.write(f"Error searching for {filename}: {e}")
            
    return all_links

In [ ]:
download_urls = get_granule_links(bad_files)

In [ ]:
len(download_urls)

In [ ]:
output_file = 'redownload_vD_2026_03_27.json'

with open(output_file, 'w') as f:
    json.dump(download_urls, f, indent=4)

print(f"Successfully saved {len(download_urls)} entries to {output_file}")

In [ ]:
with open('/nas/cee-ice/data/SWOT/redownload_vD_2026_03_27.json', 'r') as f:
    data_list = json.load(f)

In [ ]:
earthaccess.download(download_urls[0:5000],
                     f'/nas/cee-ice/data/SWOT/PIXC_vD',
                     threads=4)

In [ ]:
earthaccess.download(links_list_clean[5000:len(data_list)],
                     f'/nas/cee-ice/data/SWOT/PIXC_vD',
                     threads=4)

In [ ]:
# Usage:
# redownload_corrupted(bad_files, swot_dir)

In [ ]:
len(bad_files)

### Organize

In [ ]:
granules = pd.DataFrame(data=pixc_files, columns=['granule'])

In [ ]:
pixc_files[0]

In [ ]:
pixc_files[0][-55:-52]

In [ ]:
# FOR PRIOR
granules.loc[:,'cycle'] = granules['granule'].str.slice(-55,-52)
granules.loc[:,'pass'] = granules['granule'].str.slice(-51,-48)
granules.loc[:,'tile'] = granules['granule'].str.slice(-47,-43)
granules.loc[:,'version'] = granules['granule'].str.slice(-10,-6)
granules.loc[:,'counter'] = granules['granule'].str.slice(-5,-3)

In [ ]:
granules = granules.sort_values(by=['cycle', 'pass', 'tile', 'version', 'counter'],
                                ascending=[True, True, True, True, False])

In [ ]:
# Unique combos
granules[['pass', 'tile']].drop_duplicates()

In [ ]:
granules.drop_duplicates(subset=['cycle', 'pass', 'tile'], keep='first')

In [ ]:
# script
import os
import shutil
import re
from pathlib import Path
from tqdm import tqdm

source_dir = "/path/to/source"
dest_root = "/path/to/destination"

# Pattern matches SWOT_L2_HR_PIXC_{cycle}_{pass}_{tile}...
pattern = re.compile(r"SWOT_L2_HR_PIXC_\d{3}_(\d{3})_")

files = [f for f in os.listdir(source_dir) if f.endswith(".nc")]

for filename in tqdm(files, desc="Organizing SWOT files"):
    match = pattern.search(filename)
    if match:
        pass_num = match.group(1)
        target_dir = Path(dest_root) / pass_num
        
        # Create the pass directory if it doesn't exist
        target_dir.mkdir(parents=True, exist_ok=True)
        
        # Use shutil.copy2 to preserve metadata (like rsync -a)
        shutil.copy2(Path(source_dir) / filename, target_dir / filename)


In [ ]:
# bash

for f in SWOT_L2_HR_PIXC_*.nc; do
    # Extracts '315' from the filename
    pass_num=$(echo "$f" | cut -d'_' -f6)
    
    mkdir -p "/path/to/destination/$pass_num"
    cp -p "$f" "/path/to/destination/$pass_num/"
done